In [1]:
%pip install transformers datasets accelerate
%pip install huggingface_hub

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import load_dataset
from huggingface_hub import login

d:\finetune llm\gpt2tuned\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
login()

ImportError: The `notebook_login` function can only be used in a notebook (Jupyter or Colab) and you need the `ipywidgets` module: `pip install ipywidgets`.

In [4]:
dataset = load_dataset("Amod/mental_health_counseling_conversations")
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['Context', 'Response'],
        num_rows: 3512
    })
})


In [5]:
def concatenate_columns(examples):
    return {"text": examples["Context"] + " " + examples["Response"]}

In [6]:
dataset = dataset.map(concatenate_columns, remove_columns=["Context", "Response"])
print(dataset["train"][0])

{'text': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone? If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is somehow terrible.Bad fee

In [7]:
train_test_split = dataset["train"].train_test_split(test_size=0.1)
train_dataset = train_test_split["train"]
eval_dataset = train_test_split["test"]
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")

Training samples: 3160
Validation samples: 352


In [8]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2",cache_dir=None, force_download=True)
tokenizer.pad_token = tokenizer.eos_token

In [10]:
def tokenize_function(examples):
    return tokenizer(examples['text'], truncation=True, padding="max_length", max_length=128)

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(tokenize_function, batched=True)

print(f"Tokenized training samples: {len(tokenized_train_dataset)}")
print(f"Tokenized validation samples: {len(tokenized_eval_dataset)}")

Map: 100%|██████████| 352/352 [00:01<00:00, 301.45 examples/s]

Tokenized training samples: 3160
Tokenized validation samples: 352


In [11]:
model = GPT2LMHeadModel.from_pretrained("gpt2")

In [12]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [13]:
training_args = TrainingArguments(
    output_dir="./gpt2-fine-tuned",
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    save_steps=500,
    save_total_limit=2,
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    fp16=True
)


d:\finetune llm\gpt2tuned\Lib\site-packages\transformers\training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    data_collator=data_collator,
)

In [15]:
trainer.train()

  0%|          | 10/2370 [00:36<2:28:09,  3.77s/it]

{'loss': 3.5432, 'grad_norm': 12.503683090209961, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.01}


  1%|          | 20/2370 [01:28<3:35:40,  5.51s/it]

{'loss': 3.4991, 'grad_norm': 12.78115463256836, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.03}


  1%|▏         | 30/2370 [02:30<3:39:29,  5.63s/it]

...

100%|██████████| 2370/2370 [2:49:45<00:00,  4.38s/it]

{'loss': 2.1367, 'grad_norm': 8.969375610351562, 'learning_rate': 0.0, 'epoch': 3.0}


                                                     
100%|██████████| 2370/2370 [2:51:33<00:00,  4.34s/it]

{'eval_loss': 2.1565308570861816, 'eval_runtime': 107.791, 'eval_samples_per_second': 3.266, 'eval_steps_per_second': 0.816, 'epoch': 3.0}
{'train_runtime': 10293.5721, 'train_samples_per_second': 0.921, 'train_steps_per_second': 0.23, 'train_loss': 2.500353047817568, 'epoch': 3.0}


TrainOutput(global_step=2370, training_loss=2.500353047817568, metrics={'train_runtime': 10293.5721, 'train_samples_per_second': 0.921, 'train_steps_per_second': 0.23, 'total_flos': 619262115840000.0, 'train_loss': 2.500353047817568, 'epoch': 3.0})

In [16]:
results = trainer.evaluate()
print(results)

100%|██████████| 88/88 [01:41<00:00,  1.15s/it]

{'eval_loss': 2.1565308570861816, 'eval_runtime': 102.9479, 'eval_samples_per_second': 3.419, 'eval_steps_per_second': 0.855, 'epoch': 3.0}


In [17]:
model.save_pretrained("./gpt2-fine-tuned")
tokenizer.save_pretrained("./gpt2-fine-tuned")

('./gpt2-fine-tuned\\tokenizer_config.json',
 './gpt2-fine-tuned\\special_tokens_map.json',
 './gpt2-fine-tuned\\vocab.json',
 './gpt2-fine-tuned\\merges.txt',
 './gpt2-fine-tuned\\added_tokens.json')